# Système de recommandation d'articles (My Content)

MVP d'un système de recommandation d'articles de presse. Ce notebook couvre l'exploration du
jeu de données Globo.com, l'élaboration des deux modèles de recommandation, et la préparation
des artefacts déployés sur Azure Functions.

Besoin fonctionnel : *en tant qu'utilisateur de l'application, je reçois une sélection de cinq articles.*

**Sommaire**

1. Exploration des données
2. Élaboration d'un modèle Content-Based Filtering
3. Élaboration d'un modèle de type Collaborative Filtering
4. Préparation des artefacts pour Azure

# 1. Exploration des données

Objectif : établir les chiffres qui conditionnent les choix de modélisation, à savoir la taille
du catalogue, la longueur des historiques, la part du catalogue réellement consultée, et
l'ampleur du cold start.

## Environnement et fichiers

In [1]:
import sys, pandas as pd
print(sys.executable)
print(pd.__version__)

/home/klt/prjet-10_opclrm/.venv310/bin/python
2.3.3


In [2]:
from pathlib import Path

DATA = Path("news-portal-user-interactions-by-globocom")
assert DATA.exists(), f"introuvable : {DATA.resolve()}"

for p in sorted(DATA.iterdir()):
    if p.is_file():
        print(f"{p.name} : {p.stat().st_size/1e6:.1f} Mo")
    else:
        n = len(list(p.rglob('*.csv')))
        print(f"[dir] {p.name} : {n} CSV")

articles_embeddings.pickle : 364.0 Mo
articles_metadata.csv : 11.1 Mo
[dir] clicks : 385 CSV
clicks_sample.csv : 0.1 Mo


## Catalogue d'articles

`articles_metadata.csv` relie chaque `article_id` à sa catégorie, sa date de publication et son
nombre de mots.

In [3]:
import pandas as pd

meta = pd.read_csv(DATA / "articles_metadata.csv")
print(meta.shape)
print(meta.dtypes)
meta.head()

(364047, 5)
article_id       int64
category_id      int64
created_at_ts    int64
publisher_id     int64
words_count      int64
dtype: object


,article_id,category_id,created_at_ts,publisher_id,words_count
0,0,0,1513144419000,0,168
1,1,1,1405341936000,0,189
2,2,1,1408667706000,0,250
3,3,1,1408468313000,0,230
4,4,1,1407071171000,0,162


In [4]:
print("articles :", meta.article_id.nunique())
print("catégories :", meta.category_id.nunique())
print("publishers :", meta.publisher_id.nunique())
print("\nmots par article :")
print(meta.words_count.describe())

articles : 364047
catégories : 461
publishers : 1

mots par article :
count    364047.000000
mean        190.897727
std          59.502766
min           0.000000
25%         159.000000
50%         186.000000
75%         218.000000
max        6690.000000
Name: words_count, dtype: float64


461 catégories, exploitables comme variable de contenu. La colonne `publisher_id` ne prend
qu'une seule valeur et n'apporte donc aucune information.

## Embeddings des articles

Le fichier `articles_embeddings.pickle` contient une représentation vectorielle de chaque
article, produite en amont par un modèle de langage. Il faut vérifier que l'indice de ligne
correspond bien à l'`article_id`.

In [5]:
import pickle, numpy as np

with open(DATA / "articles_embeddings.pickle", "rb") as f:
    emb = pickle.load(f)

emb = np.asarray(emb)
print("shape :", emb.shape)
print("dtype :", emb.dtype)
print("mémoire :", emb.nbytes / 1e6, "Mo")
print("aligné avec meta :", emb.shape[0] == len(meta))

shape : (364047, 250)
dtype : float32
mémoire : 364.047 Mo
aligné avec meta : True


L'indice de ligne correspond directement à l'`article_id`, ce qui évite toute table de
correspondance par la suite.

En revanche, les embeddings pèsent 364 Mo en float32, bien au-delà de ce qui tient dans les
limites du plan Azure gratuit. Une réduction de dimension par ACP sera donc nécessaire avant
tout déploiement (partie 4).

## Interactions utilisateurs

`clicks_sample.csv` donne le schéma des colonnes avant de traiter le dossier complet.

In [6]:
sample = pd.read_csv(DATA / "clicks_sample.csv")
print(sample.shape)
print(sample.columns.tolist())
sample.head()

(1883, 12)
['user_id', 'session_id', 'session_start', 'session_size', 'click_article_id', 'click_timestamp', 'click_environment', 'click_deviceGroup', 'click_os', 'click_country', 'click_region', 'click_referrer_type']


,user_id,session_id,session_start,session_size,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,0,1506825423271737,1506825423000,2,157541,1506826828020,4,3,20,1,20,2
1,0,1506825423271737,1506825423000,2,68866,1506826858020,4,3,20,1,20,2
2,1,1506825426267738,1506825426000,2,235840,1506827017951,4,1,17,1,16,2
3,1,1506825426267738,1506825426000,2,96663,1506827047951,4,1,17,1,16,2
4,2,1506825435299739,1506825435000,2,119592,1506827090575,4,1,17,1,24,2


### Concaténation des fichiers de clics

Les clics sont livrés en 385 fichiers CSV, un par tranche horaire. Le système de recommandation
a besoin de l'historique complet de chaque utilisateur, donc d'un fichier unique : les clics
d'une même personne sont répartis sur plusieurs fichiers selon le moment de sa visite.

Le résultat est sauvegardé en Parquet, ce qui évite de relire les 385 CSV à chaque ouverture du
notebook et conserve les types de colonnes.

In [7]:
from pathlib import Path
import pandas as pd

files = sorted((DATA / "clicks").rglob("*.csv"))
print(len(files), "fichiers")

clicks = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
print(clicks.shape)

Path("data").mkdir(exist_ok=True)
clicks.to_parquet("data/clicks_all.parquet", index=False)

385 fichiers
(2988181, 12)


In [8]:
n_users = clicks.user_id.nunique()
n_art_clicked = clicks.click_article_id.nunique()

print("interactions :", len(clicks))
print("utilisateurs :", n_users)
print("articles cliqués :", n_art_clicked, f"({n_art_clicked/len(meta):.1%} du catalogue)")
print("densité :", len(clicks) / (n_users * n_art_clicked))
print("\nclics par utilisateur :")
print(clicks.groupby("user_id").size().describe())

interactions : 2988181
utilisateurs : 322897
articles cliqués : 46033 (12.6% du catalogue)
densité : 0.00020103589647179989

clics par utilisateur :
count    322897.000000
mean          9.254285
std          14.946358
min           2.000000
25%           2.000000
50%           4.000000
75%          10.000000
max        1232.000000
dtype: float64


Trois chiffres orientent la suite du projet.

**Densité de 0,02 %.** La matrice utilisateurs x articles est vide à 99,98 %. Elle ne sera jamais
manipulée sous forme dense, mais en format creux (`scipy.sparse`).

**Médiane de 4 clics par utilisateur**, premier quartile à 2. La distribution est très asymétrique
(moyenne 9,25, maximum 1232) : une minorité d'utilisateurs très actifs coexiste avec une masse de
visiteurs occasionnels. Le filtrage collaboratif dispose donc de peu de signal pour la majorité de
la population.

**46 033 articles cliqués sur 364 047, soit 12,6 % du catalogue.** Près de 9 articles sur 10 n'ont
jamais été consultés et restent hors de portée d'un modèle collaboratif, quel que soit son
entraînement. Seule une approche par le contenu peut les faire remonter.

## Dimension temporelle

La fenêtre d'observation détermine où placer la coupure entre entraînement et test.

In [9]:
clicks["click_dt"] = pd.to_datetime(clicks.click_timestamp, unit="ms")
print("du", clicks.click_dt.min(), "au", clicks.click_dt.max())
print("durée :", clicks.click_dt.max() - clicks.click_dt.min())

meta["created_dt"] = pd.to_datetime(meta.created_at_ts, unit="ms")
print("\narticles créés du", meta.created_dt.min(), "au", meta.created_dt.max())

du 2017-10-01 03:00:00.026000 au 2017-11-13 20:04:14.886000
durée : 43 days 17:04:14.860000

articles créés du 2006-09-27 11:14:35 au 2018-03-13 12:12:30


In [10]:
par_jour = clicks.set_index("click_dt").resample("D").size()
print(par_jour)

click_dt
2017-10-01     94056
2017-10-02    303177
2017-10-03    261159
2017-10-04    215415
2017-10-05    190003
2017-10-06    207646
2017-10-07    139323
2017-10-08    108110
2017-10-09    248208
2017-10-10    282391
2017-10-11    238969
2017-10-12    121467
2017-10-13    180723
2017-10-14     95216
2017-10-15     92163
2017-10-16    189779
2017-10-17     19664
2017-10-18       272
2017-10-19       125
2017-10-20       122
2017-10-21        23
2017-10-22        32
2017-10-23        40
2017-10-24        29
2017-10-25        18
2017-10-26        12
2017-10-27         7
2017-10-28         2
2017-10-29         0
2017-10-30        12
2017-10-31         4
2017-11-01         6
2017-11-02         0
2017-11-03         2
2017-11-04         2
2017-11-05         0
2017-11-06         0
2017-11-07         2
2017-11-08         0
2017-11-09         0
2017-11-10         0
2017-11-11         0
2017-11-12         0
2017-11-13         2
Freq: D, dtype: int64


Le volume s'effondre à partir du 17 octobre. Du 1er au 16, les journées se situent entre 90 000 et
300 000 clics. Le 17 tombe à 19 664, le 18 à 272, puis les journées suivantes oscillent entre 0 et
quelques dizaines de clics jusqu'au 13 novembre.

Ce n'est ni une baisse d'activité réelle (aucun site ne perd 99,9 % de son audience du jour au
lendemain), ni un arrêt net puisque des lignes continuent d'arriver. Les cellules suivantes
cherchent à caractériser ces interactions résiduelles.

Autre point notable : les articles ont été créés entre 2006 et mars 2018, alors que les clics
s'arrêtent en octobre 2017. Il existe donc des articles postérieurs à toute interaction observée,
qu'un modèle collaboratif ne pourra jamais recommander. C'est une illustration directe du cold
start article.

In [11]:
tardif = clicks[clicks.click_dt >= "2017-10-18"]
print(tardif.user_id.nunique(), "users |", len(tardif), "clics")
print(tardif.click_environment.value_counts())
print(tardif.click_country.value_counts().head())

134 users | 712 clics
click_environment
4    712
Name: count, dtype: int64
click_country
1     655
10     37
6      11
2       5
9       2
Name: count, dtype: int64


In [12]:
print(clicks[clicks.click_dt < "2017-10-18"].click_environment.value_counts(normalize=True))

click_environment
4    0.971982
2    0.026692
1    0.001326
Name: proportion, dtype: float64


In [13]:
tardif = clicks[clicks.click_dt >= "2017-10-18"]
print(tardif.groupby("user_id").size().describe())
print("\nusers tardifs déjà vus avant :", 
      len(set(tardif.user_id) & set(clicks[clicks.click_dt < "2017-10-18"].user_id)))

count    134.000000
mean       5.313433
std        4.489617
min        2.000000
25%        2.000000
50%        4.000000
75%        6.750000
max       22.000000
dtype: float64

users tardifs déjà vus avant : 129


L'environnement 4 représente déjà 97,2 % des clics avant le 17 octobre : son homogénéité sur la
période tardive n'apprend donc rien. En revanche, 129 des 134 utilisateurs tardifs, soit 96 %,
étaient déjà présents avant le 17.

Il ne s'agit pas d'une nouvelle population, mais d'un sous-ensemble d'utilisateurs déjà observés
qui continue d'être suivi. Cela suggère un changement de périmètre de collecte plutôt qu'une
évolution du comportement.

Ces 712 clics représentent 0,02 % du jeu de données. La cause exacte demanderait de connaître le
protocole de collecte de Globo.com, mais le fait établi suffit : le signal change de nature au
17 octobre.

**Période retenue : 1er au 16 octobre 2017.**

## Découpage train / test

Le découpage est temporel et non aléatoire : en production, le système prédit toujours le futur à
partir du passé. Un tirage aléatoire permettrait d'entraîner sur des clics du 16 pour prédire ceux
du 12, ce qui gonflerait artificiellement les résultats sans rien dire du comportement réel.

Entraînement du 1er au 13 octobre, test du 14 au 16.

In [14]:
import pandas as pd

FIN = pd.Timestamp("2017-10-17")
clicks_ok = clicks[clicks.click_dt < FIN].copy()
print("avant :", len(clicks), "→ après :", len(clicks_ok))

CUT = pd.Timestamp("2017-10-14")
train = clicks_ok[clicks_ok.click_dt < CUT]
test  = clicks_ok[clicks_ok.click_dt >= CUT]

print("\ntrain :", len(train), "|", train.user_id.nunique(), "users")
print("test  :", len(test), "|", test.user_id.nunique(), "users")

avant : 2988181 → après : 2967805

train : 2590647 | 302101 users
test  : 377158 | 95773 users


In [15]:
users_test = set(test.user_id)
users_train = set(train.user_id)

connus = users_test & users_train
print("users test connus du train :", len(connus), f"({len(connus)/len(users_test):.1%})")
print("users test en cold start   :", len(users_test - users_train))

users test connus du train : 76002 (79.4%)
users test en cold start   : 19771


79,4 % des utilisateurs actifs pendant la période de test ont un historique dans la période
d'entraînement. Les 20,6 % restants, soit 19 771 utilisateurs, arrivent sans passé : c'est le cold
start utilisateur, mesuré sur les données et non supposé.

Conséquence directe pour l'architecture : un utilisateur sur cinq ne peut pas être servi par un
modèle collaboratif. Une branche de repli est indispensable.

## Échantillon de travail

Calculer les similarités pour les 322 897 utilisateurs est hors de portée dans un cadre MVP. Le
travail porte sur 5 000 utilisateurs tirés au hasard parmi ceux ayant au moins 5 clics dans la
période d'entraînement, ce qui garantit un historique exploitable par les deux modèles.

Le tirage est fixé par une graine : l'échantillon est reproductible d'une exécution à l'autre.

In [16]:
import numpy as np

nb_clics = train.groupby("user_id").size()
eligibles = nb_clics[nb_clics >= 5].index
print("utilisateurs éligibles :", len(eligibles))

rng = np.random.default_rng(42)
echantillon = rng.choice(eligibles, size=min(5000, len(eligibles)), replace=False)
print("échantillon :", len(echantillon))

utilisateurs éligibles : 145586
échantillon : 5000


# 2. Élaboration d'un modèle Content-Based Filtering

Principe : chaque article est un point dans l'espace des embeddings, où la proximité traduit la
similarité de contenu. Le profil d'un utilisateur est construit à partir des articles qu'il a
consultés, puis on retient les articles du catalogue les plus proches de ce profil au sens de la
similarité cosinus.

Le fichier `articles_embeddings` évite tout prétraitement de type TF-IDF. Aucun entraînement n'est
nécessaire : la méthode fonctionne dès le premier article lu, et peut recommander un article que
personne n'a encore consulté.

L'énoncé laisse le choix de la stratégie de construction du profil. Deux sont comparées ici : la
moyenne des embeddings de tous les articles lus, et le dernier article consulté.

Historique de chaque utilisateur de l'échantillon.

In [17]:
hist = (train[train.user_id.isin(echantillon)]
        .groupby("user_id")["click_article_id"]
        .apply(list)
        .to_dict())

u = echantillon[0]
print("user", u, "→", len(hist[u]), "articles :", hist[u][:10])

user 99653 → 6 articles : [160417, 158536, 235230, 83893, 235440, 338350]


Les vecteurs sont ramenés à une longueur de 1. La similarité cosinus se réduit alors à un simple
produit scalaire, ce qui évite de recalculer les normes à chaque requête.

In [18]:
norms = np.linalg.norm(emb, axis=1, keepdims=True)
emb_norm = emb / np.where(norms == 0, 1, norms)
print(emb_norm.shape)

(364047, 250)


### Stratégie 1 : moyenne des embeddings des articles lus

`argpartition` évite de trier les 364 047 scores quand seuls les 5 premiers sont utiles, et
`-np.inf` écarte les articles déjà consultés.

In [19]:
def recommander(user_id, k=5):
    articles_lus = hist[user_id]
    profil = emb_norm[articles_lus].mean(axis=0)
    profil = profil / np.linalg.norm(profil)

    scores = emb_norm @ profil
    scores[articles_lus] = -np.inf

    top = np.argpartition(-scores, k)[:k]
    return top[np.argsort(-scores[top])]

In [20]:
u = echantillon[0]
reco = recommander(u)

print("A LU :")
print(meta.loc[hist[u], ["article_id", "category_id", "words_count"]])
print("\nON PROPOSE :")
print(meta.loc[reco, ["article_id", "category_id", "words_count"]])

A LU :
        article_id  category_id  words_count
160417      160417          281          173
158536      158536          281          858
235230      235230          375          262
83893        83893          174          180
235440      235440          375          211
338350      338350          437          177

ON PROPOSE :
        article_id  category_id  words_count
103066      103066          228          274
235294      235294          375          178
158543      158543          281          211
107335      107335          228          146
98292        98292          221          144


In [21]:
import numpy as np

taux = []
for u in echantillon[:200]:
    cats_lues = set(meta.loc[hist[u], "category_id"])
    cats_reco = meta.loc[recommander(u), "category_id"]
    taux.append(cats_reco.isin(cats_lues).mean())

print(f"recouvrement moyen des catégories : {np.mean(taux):.1%}")

recouvrement moyen des catégories : 72.9%


Les catégories recommandées recoupent celles des articles lus sans s'y limiter : les embeddings
mesurent la proximité de contenu, pas l'appartenance à une catégorie. Deux articles classés
différemment peuvent traiter de sujets voisins.

Le recouvrement moyen s'établit à **72,9 %** : la continuité thématique est forte, sans fermeture
complète à la découverte.

### Stratégie 2 : dernier article consulté

In [22]:
def recommander_dernier(user_id, k=5):
    lus = hist[user_id]
    dernier = lus[-1]
    scores = emb_norm @ emb_norm[dernier]
    scores[lus] = -np.inf
    top = np.argpartition(-scores, k)[:k]
    return top[np.argsort(-scores[top])]

print("moyenne :", recommander(u))
print("dernier :", recommander_dernier(u))


moyenne : [353599 279733 285777  96199 281359]
dernier : [118443 118444 148917 148058 288457]


**Stratégie retenue : la moyenne.**

La stratégie du dernier article ramène le voisinage immédiat d'un seul point. On le voit aux
identifiants consécutifs 118443 et 118444, donc des articles publiés à la suite sur des sujets
proches : le résultat est pointu mais étroit.

La moyenne agrège plusieurs centres d'intérêt et produit des recommandations plus diversifiées.
Sur un site d'actualité, le dernier article capterait mieux l'intention immédiate, au prix d'un
enfermement thématique.

Réserve méthodologique : `hist` n'est pas trié explicitement par horodatage, donc « le dernier »
de la liste n'est pas garanti chronologique. Cette stratégie n'étant pas retenue, le point n'a pas
été corrigé.

Sauvegarde des embeddings normalisés. Ce fichier de 364 Mo est trop lourd pour Azure : sa version
réduite est produite en partie 4.

In [23]:
np.save("data/emb_norm.npy", emb_norm.astype(np.float32))
print("sauvegardé :", emb_norm.nbytes / 1e6, "Mo")

sauvegardé : 364.047 Mo


# 3. Élaboration d'un modèle de type Collaborative Filtering

Principe inverse du précédent : ne regarder que les co-occurrences de lecture, jamais le contenu
des articles. Si des utilisateurs au parcours semblable ont lu un article que je n'ai pas lu, il
m'est proposé.

Exemple minimal :

| Utilisateur | Articles lus |
|---|---|
| moi | 5, 12, 30 |
| Paul | 5, 12, 30, 47 |
| Marie | 100, 200, 300 |

Paul partage trois articles avec moi, Marie aucun. L'article 47 m'est donc recommandé.

Avec 322 897 utilisateurs et une médiane de 4 clics, la comparaison directe deux à deux serait à
la fois trop coûteuse et trop peu informative. La factorisation matricielle (ALS) résume chaque
utilisateur et chaque article par un petit nombre de facteurs latents, ajustés par optimisation
pour reproduire les clics observés.

## Rating implicite

Le jeu de données ne comporte aucune note explicite. Le rating est donc reconstruit à partir du
nombre de clics d'un utilisateur sur un article.

La colonne `session_size` n'est pas utilisable comme rating : elle compte les clics de l'ensemble
d'une session et n'est rattachable à aucun article en particulier.

In [24]:
r = train.groupby(["user_id", "click_article_id"]).size()
print(r.value_counts().head())

1    2528127
2      27678
3       1624
4        278
5         85
Name: count, dtype: int64


98,8 % des paires utilisateur-article sont à un seul clic : le signal est binaire en pratique.

Conséquence sur le choix de l'algorithme. Une factorisation conçue pour des notes explicites
(type SVD) chercherait à prédire une valeur qui ne varie pas. ALS en feedback implicite traite les
données comme « observé / non observé », ce qui correspond exactement à la situation.

Une pondération par le nombre total de clics de l'utilisateur ne créerait aucune différence entre
ses articles, puisqu'ils seraient tous divisés par la même valeur. Elle sert à éviter qu'un
utilisateur à 1232 clics pèse 300 fois plus qu'un utilisateur à 4 dans l'apprentissage, ce qui est
une normalisation et non un enrichissement du signal.

## Construction de la matrice creuse

Les identifiants d'articles vont jusqu'à 364 047, alors que l'échantillon n'en contient qu'une
fraction. Deux dictionnaires de correspondance traduisent les identifiants réels en indices de
matrice, et inversement. `sum_duplicates` regroupe les clics répétés sur un même article, ce qui
produit le rating implicite.

In [25]:
from scipy.sparse import csr_matrix
import numpy as np

sub = train[train.user_id.isin(echantillon)]

users = np.sort(sub.user_id.unique())
items = np.sort(sub.click_article_id.unique())

u_idx = {u: i for i, u in enumerate(users)}
i_idx = {a: i for i, a in enumerate(items)}

rows = sub.user_id.map(u_idx).values
cols = sub.click_article_id.map(i_idx).values
vals = np.ones(len(sub), dtype=np.float32)

mat = csr_matrix((vals, (rows, cols)), shape=(len(users), len(items)))
mat.sum_duplicates()

print(mat.shape, "|", mat.nnz, "interactions")
print("densité :", mat.nnz / (mat.shape[0] * mat.shape[1]))

(5000, 6673) | 73992 interactions
densité : 0.0022176532294320398


Matrice 5 000 x 6 673, densité 0,22 %, bien plus dense que la matrice complète (0,02 %) puisque
l'échantillon est filtré sur les utilisateurs actifs.

Ce point invite à interpréter avec prudence toute comparaison avec le content-based :
l'échantillon est volontairement favorable au collaboratif.

## Entraînement ALS

50 facteurs latents par utilisateur et par article, 20 itérations. Chaque utilisateur et chaque
article est donc résumé par 50 nombres appris, dont le produit scalaire reconstitue au mieux les
clics observés.

In [26]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(
    factors=50,
    regularization=0.05,
    iterations=20,
    random_state=42,
)
model.fit(mat)
print("terminé")

/home/klt/prjet-10_opclrm/.venv310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/klt/prjet-10_opclrm/.venv310/lib/python3.10/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████████████████████████████████████| 20/20 [00:00<00:00, 32.66it/s]

terminé


In [27]:
def recommander_collab(user_id, k=5):
    ui = u_idx[user_id]
    ids, scores = model.recommend(ui, mat[ui], N=k, filter_already_liked_items=True)
    return [items[i] for i in ids], scores

## Comparaison des deux modèles sur un même utilisateur

In [28]:
uid = echantillon[0]

print("A LU :")
print(meta.loc[hist[uid], ["article_id", "category_id"]].to_string(index=False))

print("\nCONTENT-BASED :")
print(meta.loc[recommander(uid), ["article_id", "category_id"]].to_string(index=False))

arts, sc = recommander_collab(uid)
print("\nCOLLABORATIF :")
print(meta.loc[arts, ["article_id", "category_id"]].to_string(index=False))
print("scores :", sc.round(3))

A LU :
 article_id  category_id
     160417          281
     158536          281
     235230          375
      83893          174
     235440          375
     338350          437

CONTENT-BASED :
 article_id  category_id
     103066          228
     235294          375
     158543          281
     107335          228
      98292          221

COLLABORATIF :
 article_id  category_id
     313431          431
     156964          281
      70646          136
     156619          281
     119193          247
scores : [0.179 0.158 0.148 0.13  0.127]


L'utilisateur a lu dans les catégories 281, 375, 174 et 437.

Le content-based propose 228, 375, 281, 228 et 221 : deux catégories reprennent exactement ses
lectures.

Le collaboratif propose 431, 281, 136, 281 et 247 : une seule catégorie commune, et il s'écarte
vers 431, 136 et 247 que l'utilisateur n'a jamais consultées.

C'est le comportement attendu. Le collaboratif ne voit pas le contenu : il a repéré que des
lecteurs au parcours semblable ont consulté ces articles, et les propose sans savoir de quoi ils
traitent. D'où davantage de découverte, mais aussi plus de risque de hors-sujet.

Les scores, compris entre 0,127 et 0,179, sont faibles et resserrés, ce qui est attendu avec six
clics d'historique. Ce sont des préférences reconstruites et non des probabilités : seul leur
ordre a un sens.

## Sauvegarde des facteurs

Ces fichiers sont légers, environ 1 Mo et 1,3 Mo, et seront lus par l'Azure Function depuis le
Blob Storage.

In [29]:
import pickle, numpy as np

np.save("data/als_user_factors.npy", model.user_factors.astype(np.float32))
np.save("data/als_item_factors.npy", model.item_factors.astype(np.float32))

with open("data/als_mappings.pkl", "wb") as f:
    pickle.dump({"users": users, "items": items}, f)

print("user_factors :", model.user_factors.shape)
print("item_factors :", model.item_factors.shape)

user_factors : (5000, 50)
item_factors : (6673, 50)


# 4. Préparation des artefacts pour Azure

L'Azure Function doit pouvoir répondre à n'importe quel `user_id`, y compris inconnu. Elle
applique donc une cascade à trois branches :

1. utilisateur présent dans le modèle ALS : filtrage collaboratif ;
2. utilisateur absent d'ALS mais dont l'historique est connu : content-based ;
3. aucun historique : popularité.

Cette partie produit les fichiers dont chaque branche a besoin, sous une forme compatible avec les
limites du plan Consumption.

## Réduction de dimension par ACP

Le fichier d'embeddings pèse 364 Mo, ce qui ne tient pas dans les limites du plan gratuit. La
cellule suivante mesure la variance expliquée selon le nombre de composantes retenues, afin de
choisir le compromis en connaissance de cause.

In [30]:
from sklearn.decomposition import PCA
import numpy as np

pca_test = PCA(n_components=100, random_state=42)
pca_test.fit(emb)

cum = np.cumsum(pca_test.explained_variance_ratio_)
for n in [10, 20, 30, 50, 75, 100]:
    print(f"{n:3d} composantes : {cum[n-1]:.1%} de variance expliquée")

 10 composantes : 49.5% de variance expliquée
 20 composantes : 70.7% de variance expliquée
 30 composantes : 83.0% de variance expliquée
 50 composantes : 94.5% de variance expliquée
 75 composantes : 98.0% de variance expliquée
100 composantes : 98.7% de variance expliquée


50 composantes conservent 94,5 % de la variance pour un fichier divisé par cinq.

Ce résultat dit aussi quelque chose des données : les embeddings occupent 250 dimensions, mais
l'information réelle en occupe bien moins, puisque 20 composantes atteignent déjà 70 %. C'est
fréquent sur des embeddings de texte, souvent redondants.

Valeur retenue : 50 composantes. Les vecteurs sont renormalisés après l'ACP pour que le produit
scalaire reste une similarité cosinus.

In [31]:
N_COMP = 50

pca = PCA(n_components=N_COMP, random_state=42)
emb_red = pca.fit_transform(emb).astype(np.float32)

norms = np.linalg.norm(emb_red, axis=1, keepdims=True)
emb_red = emb_red / np.where(norms == 0, 1, norms)

print("avant :", emb.nbytes / 1e6, "Mo")
print("après :", emb_red.nbytes / 1e6, "Mo")
print("variance conservée :", f"{pca.explained_variance_ratio_.sum():.1%}")

avant : 364.047 Mo
après : 72.8094 Mo
variance conservée : 94.5%


Contrôle de qualité : les recommandations produites sur 50 dimensions doivent rester du même ordre
que celles calculées sur les embeddings complets.

In [32]:
def reco_reduit(user_id, k=5):
    lus = hist[user_id]
    profil = emb_red[lus].mean(axis=0)
    profil = profil / np.linalg.norm(profil)
    scores = emb_red @ profil
    scores[lus] = -np.inf
    top = np.argpartition(-scores, k)[:k]
    return top[np.argsort(-scores[top])]


uid = echantillon[0]
print("250 dimensions :", recommander(uid))
print(" 50 dimensions :", reco_reduit(uid))

250 dimensions : [103066 235294 158543 107335  98292]
 50 dimensions : [103066  88600  88601 103050 107335]


Sur cet utilisateur, deux des cinq articles se retrouvent dans les deux versions, dont le premier du classement. Les nouveaux venus portent des identifiants proches (88600 et 88601), donc des articles publiés à la suite : la compression déplace le classement à la marge sans changer le voisinage thématique.

In [33]:
import os

np.save("data/emb_pca50.npy", emb_red)
print(os.path.getsize("data/emb_pca50.npy") / 1e6, "Mo")

72.809528 Mo


## Historiques utilisateurs élargis

Le modèle ALS ne connaît que 5 000 utilisateurs. Pour que la branche content-based puisse traiter
des identifiants absents de ce modèle, les historiques des 50 000 utilisateurs les plus actifs sont
embarqués, limités à 20 articles chacun : au-delà, le profil moyen se dilue et le fichier grossit
sans bénéfice.

Sans ce fichier, la cascade se réduirait à deux branches et tout identifiant hors des 5 000
tomberait directement en popularité.

In [34]:
import pickle

top_users = train.groupby("user_id").size().nlargest(50000).index
hist_large = (train[train.user_id.isin(top_users)]
              .groupby("user_id")["click_article_id"]
              .apply(lambda s: list(s)[-20:])
              .to_dict())

with open("data/hist_users.pkl", "wb") as f:
    pickle.dump(hist_large, f)

print(len(hist_large), "utilisateurs |", os.path.getsize("data/hist_users.pkl") / 1e6, "Mo")

50000 utilisateurs | 4.792549 Mo


## Popularité

Troisième branche de la cascade, pour les utilisateurs sans aucun historique. Cinquante articles
sont conservés afin de garder de la marge si certains doivent être écartés.

In [35]:
top_pop = train.click_article_id.value_counts().head(50).index.tolist()

with open("data/top_popular.pkl", "wb") as f:
    pickle.dump(top_pop, f)

print(top_pop[:10])

[160974, 272143, 336221, 234698, 123909, 336223, 162655, 183176, 168623, 96210]


## Identifiants proposés dans l'application

L'interface Streamlit affiche une liste déroulante d'identifiants. Deux cents suffisent : au-delà,
la liste devient inutilisable à la souris.

In [36]:
import json

os.makedirs("app", exist_ok=True)

ids = sorted(int(u) for u in echantillon[:200])
with open("app/user_ids.json", "w") as f:
    json.dump(ids, f)

candidats = [int(u) for u in hist_large if u not in set(echantillon)][:20]
print(len(ids), "identifiants pour la liste déroulante")
print("exemples déclenchant la branche content-based :", candidats[:5])

200 identifiants pour la liste déroulante
exemples déclenchant la branche content-based : [3, 5, 6, 8, 10]


## Récapitulatif des artefacts

Ces six fichiers sont déposés dans le conteneur Blob `models`, d'où l'Azure Function les charge au
premier appel de chaque instance avant de les conserver en mémoire.

In [37]:
fichiers = [
    "emb_pca50.npy",
    "hist_users.pkl",
    "top_popular.pkl",
    "als_user_factors.npy",
    "als_item_factors.npy",
    "als_mappings.pkl",
]

for nom in fichiers:
    chemin = os.path.join("data", nom)
    if os.path.exists(chemin):
        print(f"{nom:24s} {os.path.getsize(chemin)/1e6:8.2f} Mo")
    else:
        print(f"{nom:24s}   absent")

emb_pca50.npy               72.81 Mo
hist_users.pkl               4.79 Mo
top_popular.pkl              0.00 Mo
als_user_factors.npy         1.00 Mo
als_item_factors.npy         1.33 Mo
als_mappings.pkl             0.05 Mo


Le déploiement (envoi vers le Blob Storage, code de l'Azure Function, application Streamlit) et
les schémas d'architecture MVP et cible sont décrits dans le README du dépôt.